# Neo4j AuraDB — POC Jumpstart Walkthrough

This notebook walks through the import pipeline step by step so you can verify connectivity, inspect your data, and run a test import before committing to a full load.

**Steps:**
1. Install dependencies
2. Load credentials from `.env`
3. Verify AuraDB connectivity
4. Preview a sample batch from your source
5. Run a small test import
6. Run the full import

> **Private-link users:** If `verify_connectivity()` fails, confirm your private DNS alias resolves from this environment before proceeding. On Vertex AI Notebooks, check that your VPC peering to the AuraDB region is active.

## 1. Install Dependencies

In [ ]:
# Run once. Restart the kernel after installing.
%pip install neo4j python-dotenv google-cloud-bigquery google-cloud-storage db-dtypes --quiet

## 2. Load Credentials

Copy `env.sample` to `.env` in this directory and fill in your values before running this cell.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# Confirm variables loaded — values are masked for safety
required = ["NEO4J_URI", "NEO4J_USER", "NEO4J_PASSWORD"]
for var in required:
    val = os.getenv(var)
    if val:
        masked = val[:8] + "***" if len(val) > 8 else "***"
        print(f"  {var} = {masked}")
    else:
        print(f"  ⚠️  {var} is NOT set")

## 3. Verify AuraDB Connectivity

This will immediately surface any private-link, DNS, or credential issues — before you attempt an import.

In [ ]:
from importer import Neo4jImporter

try:
    importer = Neo4jImporter()  # verify_connectivity() is called inside __init__
    print("✅ Connected successfully.")
except ConnectionError as e:
    print(f"❌ Connection failed:\n{e}")

## 4. Preview a Sample Batch

Inspect your data before writing anything to the graph. Adjust the query and source below to match your dataset.

In [ ]:
from sources.bigquery_source import BigQuerySource
# from sources.gcs_source import GCSSource  # uncomment if using GCS

# ✏️  Replace with your query / source
PREVIEW_QUERY = """
    SELECT id, name, email
    FROM `your-project.your-dataset.your-table`
    LIMIT 10
"""

source = BigQuerySource(PREVIEW_QUERY)

# Pull just the first batch and display it
first_batch = next(source.get_batches(batch_size=10))

print(f"Rows returned: {len(first_batch)}")
print(f"Columns: {list(first_batch[0].keys())}\n")
for row in first_batch:
    print(row)

### Optional: Apply Your Transform and Re-preview

If you have a `transform_fn`, verify it here before the import runs.

In [ ]:
from main import transform_part_row  # or define a lambda inline

transformed = [transform_part_row(row) for row in first_batch]
transformed = [r for r in transformed if r is not None]  # drop skipped rows

print(f"Rows after transform (skipped {len(first_batch) - len(transformed)}):")
for row in transformed:
    print(row)

## 5. Create Indexes (For MERGE)

Before importing, create indexes on the properties used in MERGE. Without them, every MERGE does a full label scan — at POC scale it's slow, at real scale it can crater the import entirely. This is one of the most common "why is my import so slow" issues.

In [ ]:
CREATE INDEX client_node_id IF NOT EXISTS FOR (p:ClientNode) ON (p.id);
CREATE INDEX supplier_code IF NOT EXISTS FOR (d:Supplier) ON (d.supplierCode);

After importing, you may want to verify indexes exist and how to verify them

In [ ]:
SHOW INDEXES;

## 6. Test Import (Small Batch)

Run a limited import first — verify node/relationship counts look right before going full scale.

In [ ]:
# ✏️  Replace with your Cypher and source
TEST_QUERY = """
    SELECT id, name, email
    FROM `your-project.your-dataset.your-table`
    LIMIT 50
"""

NODE_CYPHER = """
UNWIND $rows AS row
WITH row, datetime() AS now
MERGE (p:ClientNode {id: row.id})
ON CREATE SET p.createdAt = now, p.updatedAt = now
ON MATCH SET  p.updatedAt = now
SET p.name = row.name, p.email = row.email
"""

test_source = BigQuerySource(TEST_QUERY)
totals = importer.run_import(test_source, NODE_CYPHER, batch_size=50)

print("\n--- Test Import Summary ---")
print(f"  Nodes created:        {totals['nodes_created']}")
print(f"  Properties set:       {totals['properties_set']}")
print(f"  Relationships created: {totals['relationships_created']}")

### Verify in Neo4j

Run this cell to confirm the test nodes landed correctly.

In [ ]:
with importer.driver.session() as session:
    result = session.run("""
        MATCH (p:ClientNode)
        RETURN count(p) AS total, 
               min(p.createdAt) AS earliest,
               max(p.createdAt) AS latest
    """)
    record = result.single()
    print(f"  Total ClientNodes in graph: {record['total']}")
    print(f"  Earliest createdAt:         {record['earliest']}")
    print(f"  Latest createdAt:           {record['latest']}")

## 7. Full Import

Once the test looks good, run the full import. The importer logs progress per batch and prints a final summary.

In [ ]:
# ✏️  Replace with your full query and batch size
FULL_QUERY = """
    SELECT id, name, email
    FROM `your-project.your-dataset.your-table`
"""

full_source = BigQuerySource(FULL_QUERY)
totals = importer.run_import(full_source, NODE_CYPHER, batch_size=1000)

print("\n--- Full Import Summary ---")
print(f"  Nodes created:         {totals['nodes_created']}")
print(f"  Properties set:        {totals['properties_set']}")
print(f"  Relationships created: {totals['relationships_created']}")

## 8. Cleanup

Always close the driver when you're done.

In [ ]:
importer.close()
print("Driver closed.")

---

## Appendix: GCS CSV Import

Swap in `GCSSource` if your data is a CSV in a GCS bucket rather than a BigQuery table. Everything else is identical.

In [ ]:
from sources.gcs_source import GCSSource

SUPPLIER_CYPHER = """
UNWIND $rows AS row
MERGE (d:Supplier {supplierCode: row.code})
SET d.name = row.name, d.location = row.city
"""

# ✏️  Replace bucket_name and blob_name with your values
gcs_source = GCSSource(bucket_name="my-data-landing", blob_name="suppliers.csv")

# Preview first
first_batch = next(gcs_source.get_batches(batch_size=5))
print(f"Columns: {list(first_batch[0].keys())}")
for row in first_batch:
    print(row)

In [ ]:
# Run the GCS import
with Neo4jImporter() as gcs_importer:
    totals = gcs_importer.run_import(
        source=gcs_source,
        cypher_query=SUPPLIER_CYPHER,
        batch_size=500,
        transform_fn=lambda r: {**r, "name": r["name"].strip()},
    )
    print(f"\nSuppliers merged: {totals['nodes_created']} created, "
          f"{totals['properties_set']} properties set")